# SRGD Real-ESRGAN Kaggle Notebook (T4 x2)

这个 notebook 把仓库的命令行流程改成 Kaggle 可直接运行的版本，目标环境是 **GPU T4 x2**。

Kaggle 运行前请在右侧设置里确认：

- Accelerator: `GPU T4 x2`
- Internet: `On`
- Add data: 加入 SRGD / Super Resolution in Video Games 数据集

默认 `RUN_MODE = "smoke"`，会先用少量样本和很短迭代检查整条链路。确认没问题后，把它改成 `"full"` 再跑完整实验。

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil
import textwrap

KAGGLE = Path('/kaggle').exists()
WORK_ROOT = Path('/kaggle/working') if KAGGLE else Path.cwd()
INPUT_ROOT = Path('/kaggle/input') if KAGGLE else Path.cwd() / 'data'

PROJECT_REPO_URL = 'https://github.com/scwx-yxh/deeplearning_project.git'
REAL_ESRGAN_REPO_URL = 'https://github.com/xinntao/Real-ESRGAN.git'

PROJECT_DIR = WORK_ROOT / 'deeplearning_project'
REAL_ESRGAN_DIR = WORK_ROOT / 'Real-ESRGAN'
OUTPUT_DIR = WORK_ROOT / 'outputs'
DATA_DIR = PROJECT_DIR / 'data'
CONFIG_DIR = PROJECT_DIR / 'configs'

RUN_MODE = 'smoke'  # smoke | full
RUN_FINETUNE = True
RUN_PRETRAINED_INFERENCE = True
RUN_TEST_SUBMISSION = False

COURSE_EVAL_LIMIT = 100
PAIR_LIMIT = 64 if RUN_MODE == 'smoke' else COURSE_EVAL_LIMIT
INFERENCE_LIMIT = 16 if RUN_MODE == 'smoke' else COURSE_EVAL_LIMIT
GRID_LIMIT = 6 if RUN_MODE == 'smoke' else 12
TOTAL_ITER = 100 if RUN_MODE == 'smoke' else 5_000
SAVE_FREQ = 100 if RUN_MODE == 'smoke' else 1_000
GT_SIZE = 128 if RUN_MODE == 'smoke' else 256
BATCH_SIZE_PER_GPU = 2  # T4 16GB: try 1-4; lower to 1 on OOM
NUM_WORKERS_PER_GPU = 2
TILE = 256
MODEL_NAME = 'RealESRGAN_x4plus'

# Fine-tuning quality knobs ---------------------------------------------------
# Repeat the dataset N times per epoch.  Raise to ~50 on small datasets so
# checkpointing and logging frequencies are meaningful.
ENLARGE_RATIO = 1 if RUN_MODE == 'smoke' else 50
# Train generator-only for this many steps before enabling the discriminator.
# Lets the generator stabilise before adversarial pressure starts.
D_INIT_ITERS = 0 if RUN_MODE == 'smoke' else 1000
# LR warmup steps (0 = disabled for smoke; ramp up from 0 -> base LR in full).
WARMUP_ITER = 0 if RUN_MODE == 'smoke' else 500
# Enable 90-degree rotation augmentation (free quality boost for small datasets).
USE_ROT = True
# GAN loss weight.  5e-2 is safer for fine-tuning (1e-1 is the from-scratch default).
GAN_WEIGHT = '5e-2'

# Override these paths if auto-detection fails (see Cell 3).
DATA_ROOT_OVERRIDE = None
LR_DIR_OVERRIDE = None
HR_DIR_OVERRIDE = None
TEST_LR_DIR_OVERRIDE = None
SAMPLE_SUBMISSION_OVERRIDE = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print('\n$ ' + ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

## 1. Clone repos and install dependencies

Kaggle 每次启动都是干净环境，所以这里会自动 clone 本项目和官方 Real-ESRGAN，并安装依赖。若你把本仓库作为 Kaggle dataset 上传，也可以直接把 `PROJECT_DIR` 改到对应目录。

In [ ]:
if not (PROJECT_DIR / 'requirements.txt').exists():
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run(['git', 'clone', PROJECT_REPO_URL, PROJECT_DIR])
else:
    print(f'Using existing project repo: {PROJECT_DIR}')

if not (REAL_ESRGAN_DIR / 'requirements.txt').exists():
    if REAL_ESRGAN_DIR.exists():
        shutil.rmtree(REAL_ESRGAN_DIR)
    run(['git', 'clone', '--depth', '1', REAL_ESRGAN_REPO_URL, REAL_ESRGAN_DIR])
else:
    print(f'Using existing Real-ESRGAN repo: {REAL_ESRGAN_DIR}')

DATA_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle generally already has torch/torchvision. Avoid forcing a CUDA stack reinstall.
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'wheel', 'cython', 'setuptools'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', PROJECT_DIR / 'requirements.txt'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'basicsr>=1.4.2', 'facexlib>=0.2.5', 'gfpgan>=1.3.5'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REAL_ESRGAN_DIR / 'requirements.txt'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REAL_ESRGAN_DIR])

sys.path.insert(0, str(PROJECT_DIR / 'src'))

## 1.1. Apply notebook compatibility helpers

If this notebook clones the original GitHub repo, this cell injects the small Kaggle-specific helper updates used below.


In [ ]:
import base64

# Inject the latest project scripts so this notebook works even when the cloned
# GitHub repo is behind the current notebook version.

# write_realesrgan_config.py — base64-encoded to avoid escaping issues with
# the f-string/YAML template content inside the file.
_CONFIG_WRITER_B64 = (
    'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmZyb20g'
    'cGF0aGxpYiBpbXBvcnQgUGF0aAoKCmRlZiBhc19wb3NpeChwYXRoOiBQYXRoKSAtPiBzdHI6CiAg'
    'ICByZXR1cm4gcGF0aC5yZXNvbHZlKCkuYXNfcG9zaXgoKQoKCmRlZiBidWlsZF9taWxlc3RvbmVz'
    'KHRvdGFsX2l0ZXI6IGludCkgLT4gc3RyOgogICAgIyBEZWNheSBhdCA2MCUgYW5kIDkwJSBvZiB0'
    'cmFpbmluZyBzbyB0aGUgTFIgc2NoZWR1bGUgaXMgYWN0dWFsbHkgdXNlZC4KICAgIG0xID0gbWF4'
    'KDEsIHJvdW5kKHRvdGFsX2l0ZXIgKiAwLjYpKQogICAgbTIgPSBtYXgobTEgKyAxLCByb3VuZCh0'
    'b3RhbF9pdGVyICogMC45KSkKICAgIHJldHVybiBmIlt7bTF9LCB7bTJ9XSIKCgpkZWYgYnVpbGRf'
    'Y29uZmlnKGFyZ3M6IGFyZ3BhcnNlLk5hbWVzcGFjZSkgLT4gc3RyOgogICAgZGF0YV9yb290ID0g'
    'YXNfcG9zaXgoYXJncy5kYXRhX3Jvb3QpCiAgICBtZXRhX2luZm8gPSBhc19wb3NpeChhcmdzLm1l'
    'dGFfaW5mbykKICAgIHByZXRyYWluX2cgPSBhcmdzLnByZXRyYWluX2cucmVwbGFjZSgiXFwiLCAi'
    'LyIpCiAgICBwcmV0cmFpbl9kID0gYXJncy5wcmV0cmFpbl9kLnJlcGxhY2UoIlxcIiwgIi8iKQog'
    'ICAgbWlsZXN0b25lcyA9IGJ1aWxkX21pbGVzdG9uZXMoYXJncy50b3RhbF9pdGVyKQogICAgdXNl'
    'X3JvdCA9ICJUcnVlIiBpZiBhcmdzLnVzZV9yb3QgZWxzZSAiRmFsc2UiCiAgICBwcmVmZXRjaCA9'
    'IGFyZ3MucHJlZmV0Y2hfbW9kZSBpZiBhcmdzLnByZWZldGNoX21vZGUgZWxzZSAifiIKICAgIHBp'
    'bl9tZW1vcnkgPSAiICAgIHBpbl9tZW1vcnk6IFRydWVcbiIgaWYgcHJlZmV0Y2ggPT0gImN1ZGEi'
    'IGVsc2UgIiIKCiAgICAjIENsYW1wIHdhcm11cCBhbmQgZF9pbml0IHNvIHRoZXkgc3RheSB3aXRo'
    'aW4gdHJhaW5pbmcgbGVuZ3RoLgogICAgd2FybXVwX2l0ZXIgPSBtaW4oYXJncy53YXJtdXBfaXRl'
    'ciwgbWF4KDAsIGFyZ3MudG90YWxfaXRlciAtIDEpKQogICAgZF9pbml0X2l0ZXJzID0gbWluKGFy'
    'Z3MuZF9pbml0X2l0ZXJzLCBtYXgoMCwgYXJncy50b3RhbF9pdGVyIC0gMSkpCgogICAgdmFsX2Js'
    'b2NrID0gIiIKICAgIGlmIGFyZ3MudmFsX21ldGFfaW5mbzoKICAgICAgICB2YWxfbWV0YSA9IGFz'
    'X3Bvc2l4KGFyZ3MudmFsX21ldGFfaW5mbykKICAgICAgICB2YWxfYmxvY2sgPSBmIiIiCiAgdmFs'
    'OgogICAgbmFtZTogU1JHRF92YWwKICAgIHR5cGU6IFJlYWxFU1JHQU5QYWlyZWREYXRhc2V0CiAg'
    'ICBkYXRhcm9vdF9ndDoge2RhdGFfcm9vdH0KICAgIGRhdGFyb290X2xxOiB7ZGF0YV9yb290fQog'
    'ICAgbWV0YV9pbmZvOiB7dmFsX21ldGF9CiAgICBpb19iYWNrZW5kOgogICAgICB0eXBlOiBkaXNr'
    'CiAgICBndF9zaXplOiB+CiAgICB1c2VfaGZsaXA6IEZhbHNlCiAgICB1c2Vfcm90OiBGYWxzZQog'
    'ICAgdXNlX3NodWZmbGU6IGZhbHNlCiAgICBudW1fd29ya2VyX3Blcl9ncHU6IDIKICAgIGJhdGNo'
    'X3NpemVfcGVyX2dwdTogMQogICAgZGF0YXNldF9lbmxhcmdlX3JhdGlvOiAxCiAgICBwcmVmZXRj'
    'aF9tb2RlOiB+CiIiIgoKICAgIHJldHVybiBmIiIiIyBSZWFsLUVTUkdBTiBwYWlyZWQtZGF0YSBm'
    'aW5lLXR1bmluZyBjb25maWcgZm9yIHRoZSBTUkdEIGNvdXJzZSBwcm9qZWN0LgojIEdlbmVyYXRl'
    'ZCBieSBzY3JpcHRzL3dyaXRlX3JlYWxlc3JnYW5fY29uZmlnLnB5LgoKbmFtZTogZmluZXR1bmVf'
    'UmVhbEVTUkdBTng0cGx1c19TUkdEX3BhaXJkYXRhCm1vZGVsX3R5cGU6IFJlYWxFU1JHQU5Nb2Rl'
    'bApzY2FsZTogNApudW1fZ3B1OiBhdXRvCm1hbnVhbF9zZWVkOiAwCgpsMV9ndF91c206IFRydWUK'
    'cGVyY2VwX2d0X3VzbTogVHJ1ZQpnYW5fZ3RfdXNtOiBGYWxzZQpoaWdoX29yZGVyX2RlZ3JhZGF0'
    'aW9uOiBGYWxzZQoKZGF0YXNldHM6CiAgdHJhaW46CiAgICBuYW1lOiBTUkdECiAgICB0eXBlOiBS'
    'ZWFsRVNSR0FOUGFpcmVkRGF0YXNldAogICAgZGF0YXJvb3RfZ3Q6IHtkYXRhX3Jvb3R9CiAgICBk'
    'YXRhcm9vdF9scToge2RhdGFfcm9vdH0KICAgIG1ldGFfaW5mbzoge21ldGFfaW5mb30KICAgIGlv'
    'X2JhY2tlbmQ6CiAgICAgIHR5cGU6IGRpc2sKICAgIGd0X3NpemU6IHthcmdzLmd0X3NpemV9CiAg'
    'ICB1c2VfaGZsaXA6IFRydWUKICAgIHVzZV9yb3Q6IHt1c2Vfcm90fQogICAgdXNlX3NodWZmbGU6'
    'IHRydWUKICAgIG51bV93b3JrZXJfcGVyX2dwdToge2FyZ3Mud29ya2Vyc30KICAgIGJhdGNoX3Np'
    'emVfcGVyX2dwdToge2FyZ3MuYmF0Y2hfc2l6ZX0KICAgIGRhdGFzZXRfZW5sYXJnZV9yYXRpbzog'
    'e2FyZ3MuZW5sYXJnZV9yYXRpb30KICAgIHByZWZldGNoX21vZGU6IHtwcmVmZXRjaH0Ke3Bpbl9t'
    'ZW1vcnkucnN0cmlwKCl9Cnt2YWxfYmxvY2t9Cm5ldHdvcmtfZzoKICB0eXBlOiBSUkRCTmV0CiAg'
    'bnVtX2luX2NoOiAzCiAgbnVtX291dF9jaDogMwogIG51bV9mZWF0OiA2NAogIG51bV9ibG9jazogMjMK'
    'ICBudW1fZ3Jvd19jaDogMzIKCm5ldHdvcmtfZDoKICB0eXBlOiBVTmV0RGlzY3JpbWluYXRvclNO'
    'CiAgbnVtX2luX2NoOiAzCiAgbnVtX2ZlYXQ6IDY0CiAgc2tpcF9jb25uZWN0aW9uOiBUcnVlCgpw'
    'YXRoOgogIHByZXRyYWluX25ldHdvcmtfZzoge3ByZXRyYWluX2d9CiAgcGFyYW1fa2V5X2c6IHBh'
    'cmFtc19lbWEKICBzdHJpY3RfbG9hZF9nOiB0cnVlCiAgcHJldHJhaW5fbmV0d29ya19kOiB7cHJl'
    'dHJhaW5fZH0KICBwYXJhbV9rZXlfZDogcGFyYW1zCiAgc3RyaWN0X2xvYWRfZDogdHJ1ZQogIHJl'
    'c3VtZV9zdGF0ZTogfgoKdHJhaW46CiAgZW1hX2RlY2F5OiAwLjk5OQogIG9wdGltX2c6CiAgICB0'
    'eXBlOiBBZGFtCiAgICBscjogISFmbG9hdCB7YXJncy5scl9nfQogICAgd2VpZ2h0X2RlY2F5OiAw'
    'CiAgICBiZXRhczogWzAuOSwgMC45OV0KICBvcHRpbV9kOgogICAgdHlwZTogQWRhbQogICAgbHI6'
    'ICEhZmxvYXQge2FyZ3MubHJfZH0KICAgIHdlaWdodF9kZWNheTogMAogICAgYmV0YXM6IFswLjks'
    'IDAuOTldCiAgc2NoZWR1bGVyOgogICAgdHlwZTogTXVsdGlTdGVwTFIKICAgIG1pbGVzdG9uZXM6'
    'IHttaWxlc3RvbmVzfQogICAgZ2FtbWE6IDAuNQogIHRvdGFsX2l0ZXI6IHthcmdzLnRvdGFsX2l0'
    'ZXJ9CiAgd2FybXVwX2l0ZXI6IHt3YXJtdXBfaXRlcn0KICBwaXhlbF9vcHQ6CiAgICB0eXBlOiBM'
    'MUxvc3MKICAgIGxvc3Nfd2VpZ2h0OiAxLjAKICAgIHJlZHVjdGlvbjogbWVhbgogIHBlcmNlcHR1'
    'YWxfb3B0OgogICAgdHlwZTogUGVyY2VwdHVhbExvc3MKICAgIGxheWVyX3dlaWdodHM6CiAgICAg'
    'IGNvbnYxXzI6IDAuMQogICAgICBjb252Ml8yOiAwLjEKICAgICAgY29udjNfNDogMQogICAgICBj'
    'b252NF80OiAxCiAgICAgIGNvbnY1XzQ6IDEKICAgIHZnZ190eXBlOiB2Z2cxOQogICAgdXNlX2lu'
    'cHV0X25vcm06IHRydWUKICAgIHBlcmNlcHR1YWxfd2VpZ2h0OiAhIWZsb2F0IDEuMAogICAgc3R5'
    'bGVfd2VpZ2h0OiAwCiAgICByYW5nZV9ub3JtOiBmYWxzZQogICAgY3JpdGVyaW9uOiBsMQogIGdh'
    'bl9vcHQ6CiAgICB0eXBlOiBHQU5Mb3NzCiAgICBnYW5fdHlwZTogdmFuaWxsYQogICAgcmVhbF9s'
    'YWJlbF92YWw6IDEuMAogICAgZmFrZV9sYWJlbF92YWw6IDAuMAogICAgbG9zc193ZWlnaHQ6ICEh'
    'ZmxvYXQge2FyZ3MuZ2FuX3dlaWdodH0KICBuZXRfZF9pdGVyczogMQogIG5ldF9kX2luaXRfaXRl'
    'cnM6IHtkX2luaXRfaXRlcnN9Cgpsb2dnZXI6CiAgcHJpbnRfZnJlcTogMTAwCiAgc2F2ZV9jaGVj'
    'a3BvaW50X2ZyZXE6ICEhZmxvYXQge2FyZ3Muc2F2ZV9mcmVxfQogIHVzZV90Yl9sb2dnZXI6IHRy'
    'dWUKICB3YW5kYjoKICAgIHByb2plY3Q6IH4KICAgIHJlc3VtZV9pZDogfgoKZGlzdF9wYXJhbXM6'
    'CiAgYmFja2VuZDogbmNjbAogIHBvcnQ6IDI5NTAwCiIiIgoKCmRlZiBwYXJzZV9hcmdzKCkgLT4g'
    'YXJncGFyc2UuTmFtZXNwYWNlOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIo'
    'ZGVzY3JpcHRpb249IldyaXRlIGEgUmVhbC1FU1JHQU4gcGFpcmVkLWRhdGEgZmluZS10dW5pbmcg'
    'WUFNTC4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhLXJvb3QiLCB0eXBlPVBhdGgs'
    'IHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1ldGEtaW5mbyIsIHR5'
    'cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0Iiwg'
    'dHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS12YWwt'
    'bWV0YS1pbmZvIiwgdHlwZT1QYXRoLCBkZWZhdWx0PU5vbmUsIGhlbHA9Ik9wdGlvbmFsIHZhbCBt'
    'ZXRhX2luZm8gdHh0IGZvciBwZXJpb2RpYyBldmFsLiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50'
    'KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgcGFyc2VyLmFkZF9hcmd1'
    'bWVudCgiLS13b3JrZXJzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikKICAgIHBhcnNlci5hZGRfYXJn'
    'dW1lbnQoIi0tZ3Qtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTI1NikKICAgIHBhcnNlci5hZGRf'
    'YXJndW1lbnQoIi0tdG90YWwtaXRlciIsIHR5cGU9aW50LCBkZWZhdWx0PTIwMDAwKQogICAgcGFy'
    'c2VyLmFkZF9hcmd1bWVudCgiLS1zYXZlLWZyZXEiLCB0eXBlPWludCwgZGVmYXVsdD0xMDAwKQog'
    'ICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13YXJtdXAtaXRlciIsIHR5cGU9aW50LCBkZWZhdWx0'
    'PTUwMCwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iTFIgd2FybXVwIHN0ZXBzLiBTZXQg'
    'MCB0byBkaXNhYmxlLiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWQtaW5pdC1pdGVycyIs'
    'IHR5cGU9aW50LCBkZWZhdWx0PTEwMDAsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IlRy'
    'YWluIGdlbmVyYXRvciBvbmx5IGZvciB0aGlzIG1hbnkgc3RlcHMgYmVmb3JlIGVuYWJsaW5nIHRo'
    'ZSBkaXNjcmltaW5hdG9yLiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVubGFyZ2UtcmF0'
    'aW8iLCB0eXBlPWludCwgZGVmYXVsdD0xLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJk'
    'YXRhc2V0X2VubGFyZ2VfcmF0aW86IHJlcGVhdCB0aGUgZGF0YXNldCBOIHRpbWVzIHBlciBlcG9j'
    'aC4gIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICJVc2UgZS5nLiAxMDAgd2hlbiB0aGUg'
    'ZGF0YXNldCBpcyBzbWFsbC4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS11c2Utcm90Iiwg'
    'YWN0aW9uPSJzdG9yZV90cnVlIiwgZGVmYXVsdD1UcnVlLAogICAgICAgICAgICAgICAgICAgICAg'
    'ICBoZWxwPSJFbmFibGUgOTAtZGVncmVlIHJvdGF0aW9uIGF1Z21lbnRhdGlvbi4iKQogICAgcGFy'
    'c2VyLmFkZF9hcmd1bWVudCgiLS1uby1yb3QiLCBkZXN0PSJ1c2Vfcm90IiwgYWN0aW9uPSJzdG9y'
    'ZV9mYWxzZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByZWZldGNoLW1vZGUiLCBkZWZh'
    'dWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IkJhc2ljU1IgcHJlZmV0Y2gg'
    'bW9kZTogY3VkYSwgY3B1LCBvciB+IChkaXNhYmxlZCkuIikKICAgIHBhcnNlci5hZGRfYXJndW1l'
    'bnQoIi0tbHItZyIsIGRlZmF1bHQ9IjFlLTQiLCBoZWxwPSJHZW5lcmF0b3IgbGVhcm5pbmcgcmF0'
    'ZS4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1sci1kIiwgZGVmYXVsdD0iMWUtNCIsIGhl'
    'bHA9IkRpc2NyaW1pbmF0b3IgbGVhcm5pbmcgcmF0ZS4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVu'
    'dCgiLS1sciIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iU2V0'
    'IGJvdGggLS1sci1nIGFuZCAtLWxyLWQgdG8gdGhlIHNhbWUgdmFsdWUgKHNob3J0aGFuZCkuIikK'
    'ICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZ2FuLXdlaWdodCIsIGRlZmF1bHQ9IjFlLTEiLAog'
    'ICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJHQU4gbG9zcyB3ZWlnaHQuIExvd2VyIChlLmcu'
    'IDVlLTIpIGlzIHNhZmVyIGZvciBmaW5lLXR1bmluZy4iKQogICAgIyBMZWdhY3kgYWxpYXMga2Vw'
    'dCBmb3IgYmFja3dhcmQgY29tcGF0aWJpbGl0eSB3aXRoIG9sZCBub3RlYm9vayB2ZXJzaW9ucy4K'
    'ICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tcHJldHJhaW4tZyIsCiAgICAgICAg'
    'ZGVmYXVsdD0iZXhwZXJpbWVudHMvcHJldHJhaW5lZF9tb2RlbHMvUmVhbEVTUkdBTl94NHBsdXMu'
    'cHRoIiwKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tcHJldHJhaW4t'
    'ZCIsCiAgICAgICAgZGVmYXVsdD0iZXhwZXJpbWVudHMvcHJldHJhaW5lZF9tb2RlbHMvUmVhbEVT'
    'UkdBTl94NHBsdXNfbmV0RC5wdGgiLAogICAgKQogICAgcmV0dXJuIHBhcnNlci5wYXJzZV9hcmdz'
    'KCkKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcmdzID0gcGFyc2VfYXJncygpCiAgICBpZiBh'
    'cmdzLmxyIGlzIG5vdCBOb25lOgogICAgICAgIGFyZ3MubHJfZyA9IGFyZ3MubHIKICAgICAgICBh'
    'cmdzLmxyX2QgPSBhcmdzLmxyCiAgICBhcmdzLm91dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVl'
    'LCBleGlzdF9vaz1UcnVlKQogICAgYXJncy5vdXQud3JpdGVfdGV4dChidWlsZF9jb25maWcoYXJn'
    'cyksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludChmIldyb3RlIFJlYWwtRVNSR0FOIGNvbmZp'
    'ZyB0byB7YXJncy5vdXR9IikKICAgIHByaW50KGYiICB0b3RhbF9pdGVyPXthcmdzLnRvdGFsX2l0'
    'ZXJ9ICB3YXJtdXA9e2FyZ3Mud2FybXVwX2l0ZXJ9ICBkX2luaXQ9e2FyZ3MuZF9pbml0X2l0ZXJz'
    'fSIpCiAgICBwcmludChmIiAgbWlsZXN0b25lcz17YnVpbGRfbWlsZXN0b25lcyhhcmdzLnRvdGFs'
    'X2l0ZXIpfSAgZW5sYXJnZV9yYXRpbz17YXJncy5lbmxhcmdlX3JhdGlvfSIpCgoKaWYgX19uYW1l'
    'X18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='
)
(PROJECT_DIR / 'scripts' / 'write_realesrgan_config.py').write_bytes(
    base64.b64decode(_CONFIG_WRITER_B64))

# run_realesrgan.py
(PROJECT_DIR / 'scripts' / 'run_realesrgan.py').write_text('from __future__ import annotations\n\nimport argparse\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nfrom PIL import Image\n\ntry:\n    from tqdm import tqdm\nexcept ImportError:\n    def tqdm(iterable, **_: object):\n        return iterable\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT_ROOT / "src"))\n\nfrom srgd_realesrgan.paths import IMAGE_EXTENSIONS, ensure_dir, read_pairs_csv\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--realesrgan-dir", type=Path, required=True)\n    parser.add_argument("--pairs", type=Path, required=True)\n    parser.add_argument("--out-dir", type=Path, required=True)\n    parser.add_argument("--model-name", default="RealESRGAN_x4plus")\n    parser.add_argument("--model-path", type=Path, default=None)\n    parser.add_argument("--outscale", type=float, default=4.0)\n    parser.add_argument("--tile", type=int, default=0)\n    parser.add_argument("--gpu-id", type=int, default=None)\n    parser.add_argument("--ext", default="png")\n    parser.add_argument("--fp32", action="store_true")\n    parser.add_argument("--face-enhance", action="store_true")\n    parser.add_argument("--split", default=None)\n    parser.add_argument("--limit", type=int, default=None)\n    parser.add_argument("--shuffle", action="store_true")\n    parser.add_argument("--seed", type=int, default=0)\n    return parser.parse_args()\n\n\ndef normalize_outputs(raw_dir, out_dir, rows, suffix):\n    for row in tqdm(rows, desc="Normalize outputs"):\n        pair_id = row["pair_id"]\n        matches = []\n        for ext in IMAGE_EXTENSIONS:\n            matches.extend(raw_dir.glob(f"{pair_id}_{suffix}{ext}"))\n            matches.extend(raw_dir.glob(f"{pair_id}_{suffix}{ext.upper()}"))\n        if not matches:\n            matches = list(raw_dir.glob(f"{pair_id}_*"))\n        if not matches:\n            raise FileNotFoundError(f"No output for {pair_id} in {raw_dir}")\n        Image.open(matches[0]).convert("RGB").save(out_dir / f"{pair_id}.png")\n\n\ndef main():\n    args = parse_args()\n    realesrgan_dir = args.realesrgan_dir.resolve()\n    inference_script = realesrgan_dir / "inference_realesrgan.py"\n    if not inference_script.exists():\n        raise FileNotFoundError(f"Cannot find {inference_script}")\n    rows = read_pairs_csv(args.pairs, split=args.split, limit=args.limit, shuffle=args.shuffle, seed=args.seed)\n    out_dir = ensure_dir(args.out_dir)\n    input_dir = ensure_dir(out_dir / "_lr_inputs")\n    raw_dir = ensure_dir(out_dir / "_raw_realesrgan")\n    suffix = "sr"\n    for row in tqdm(rows, desc="Stage LR inputs"):\n        src = Path(row["lr_path"])\n        shutil.copy2(src, input_dir / f"{row[\'pair_id\']}{src.suffix.lower()}")\n    cmd = [sys.executable, str(inference_script), "-n", args.model_name,\n           "-i", str(input_dir), "-o", str(raw_dir),\n           "--outscale", str(args.outscale), "--suffix", suffix,\n           "--tile", str(args.tile), "--ext", args.ext]\n    if args.model_path:\n        cmd += ["--model_path", str(args.model_path.resolve())]\n    if args.gpu_id is not None:\n        cmd += ["--gpu-id", str(args.gpu_id)]\n    if args.fp32:\n        cmd.append("--fp32")\n    if args.face_enhance:\n        cmd.append("--face_enhance")\n    print("Running:", " ".join(cmd))\n    subprocess.run(cmd, cwd=realesrgan_dir, check=True)\n    normalize_outputs(raw_dir, out_dir, rows, suffix)\n    print(f"Done: {out_dir}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')

# make_kaggle_submission.py
(PROJECT_DIR / 'scripts' / 'make_kaggle_submission.py').write_text('from __future__ import annotations\n\nimport argparse\nimport base64\nimport csv\nimport zlib\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\ntry:\n    import cv2\nexcept ImportError:\n    cv2 = None\n\nIMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"]\n\n\ndef encode_image_bgr(image):\n    flat = image.astype(np.uint8).flatten()\n    flat = np.append(flat, -1)\n    count = 1\n    rle = []\n    for i in range(1, flat.shape[0]):\n        if flat[i] == flat[i-1]:\n            count += 1\n            if count > 255:\n                rle += [int(flat[i-1]), 255]\n                count = 1\n        else:\n            rle += [int(flat[i-1]), count]\n            count = 1\n    return base64.b64encode(zlib.compress(bytes(rle), zlib.Z_BEST_COMPRESSION))\n\n\ndef find_image(images_dir, filename):\n    exact = images_dir / filename\n    if exact.exists():\n        return exact\n    stem = Path(filename).stem\n    for ext in IMAGE_EXTENSIONS:\n        c = images_dir / f"{stem}{ext}"\n        if c.exists():\n            return c\n    raise FileNotFoundError(f"No SR image for {filename} in {images_dir}")\n\n\ndef read_image_bgr(path):\n    if cv2 is not None:\n        img = cv2.imread(str(path), cv2.IMREAD_COLOR)\n        if img is None:\n            raise ValueError(f"Could not read {path}")\n        return img\n    return np.asarray(Image.open(path).convert("RGB"), dtype=np.uint8)[..., ::-1]\n\n\ndef parse_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--images-dir", type=Path, required=True)\n    p.add_argument("--out", type=Path, default=Path("submission.csv"))\n    p.add_argument("--sample-submission", type=Path, default=None)\n    p.add_argument("--id-column", default="id")\n    p.add_argument("--filename-column", default="filename")\n    p.add_argument("--rle-column", default="rle")\n    return p.parse_args()\n\n\ndef main():\n    args = parse_args()\n    images_dir = args.images_dir.resolve()\n    if args.sample_submission:\n        with args.sample_submission.open("r", newline="", encoding="utf-8") as f:\n            rows = [{"id": r.get(args.id_column, str(i)), "filename": r[args.filename_column],\n                     "image_path": find_image(images_dir, r[args.filename_column]).as_posix()}\n                    for i, r in enumerate(csv.DictReader(f))]\n    else:\n        rows = [{"id": str(i), "filename": p.name, "image_path": p.as_posix()}\n                for i, p in enumerate(sorted(p for p in images_dir.iterdir()\n                                             if p.suffix.lower() in IMAGE_EXTENSIONS))]\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    with args.out.open("w", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fieldnames=[args.id_column, args.filename_column, args.rle_column])\n        w.writeheader()\n        for row in rows:\n            w.writerow({args.id_column: row["id"], args.filename_column: row["filename"],\n                        args.rle_column: str(encode_image_bgr(read_image_bgr(row["image_path"])))})\n    print(f"Wrote {len(rows)} predictions to {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')

print('Injected: write_realesrgan_config.py, run_realesrgan.py, make_kaggle_submission.py')

In [ ]:
# Compatibility patches for Kaggle's current PyTorch/torchvision stack.
import site

def site_roots():
    site_roots = []
    try:
        site_roots.extend(site.getsitepackages())
    except AttributeError:
        pass
    try:
        site_roots.append(site.getusersitepackages())
    except AttributeError:
        pass
    return site_roots

def patch_site_file(relative_path, replacements):
    patched = []
    for root in site_roots():
        target = Path(root) / relative_path
        if not target.exists():
            continue
        text = target.read_text(encoding='utf-8')
        new_text = text
        for old, new in replacements:
            new_text = new_text.replace(old, new)
        if new_text != text:
            target.write_text(new_text, encoding='utf-8')
            patched.append(str(target))
    return patched

patched = []
patched += patch_site_file(
    Path('basicsr') / 'data' / 'degradations.py',
    [('from torchvision.transforms.functional_tensor import rgb_to_grayscale',
      'from torchvision.transforms.functional import rgb_to_grayscale')],
)
patched += patch_site_file(
    Path('basicsr') / 'utils' / 'options.py',
    [("parser.add_argument('--local_rank', type=int, default=0)",
      "parser.add_argument('--local-rank', '--local_rank', type=int, default=0)")],
)
print('Patched BasicSR files:' if patched else 'No BasicSR patch needed.', patched)
run([sys.executable, '-c', 'import torch, basicsr, realesrgan; print("imports ok", torch.__version__)'])

## 2. Check T4 x2

Notebook 无法替你切换 Kaggle accelerator；这一格只负责确认当前环境实际拿到了几张 GPU。

In [ ]:
import torch

GPU_COUNT = torch.cuda.device_count()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', GPU_COUNT)
for index in range(GPU_COUNT):
    print(index, torch.cuda.get_device_name(index))

if GPU_COUNT == 0:
    raise RuntimeError('No GPU found. In Kaggle Settings, choose GPU T4 x2 or at least one GPU.')
if KAGGLE and GPU_COUNT < 2:
    print('Warning: Kaggle is not using T4 x2 right now. Fine-tuning will fall back to the available GPU count.')

USE_GPUS = min(2, GPU_COUNT)

## 3. Discover the Kaggle dataset layout

支持常见目录：`train/lr + train/hr`、`train/270p + train/1080p`、`lq + gt`。如果你的数据目录不是这种命名，直接在第一格填写 `LR_DIR_OVERRIDE` 和 `HR_DIR_OVERRIDE`。

In [ ]:
from pathlib import Path
import os

IMAGE_EXTENSIONS = {'.bmp', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}
NAME_PAIRS = [
    ('lr', 'hr'),
    ('lq', 'gt'),
    ('low', 'high'),
    ('270p', '1080p'),
]


def has_images(folder: Path) -> bool:
    if folder is None or not folder.exists():
        return False
    return any(path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS for path in folder.rglob('*'))


def common_root(first: Path, second: Path) -> Path:
    return Path(os.path.commonpath([str(first.resolve()), str(second.resolve())]))


def candidate_roots():
    roots = []
    if KAGGLE and INPUT_ROOT.exists():
        roots.extend(sorted(path for path in INPUT_ROOT.iterdir() if path.is_dir()))
    smoke = PROJECT_DIR / 'data' / 'smoke' / 'raw'
    if smoke.exists():
        roots.append(smoke)
    return roots


def find_pair_dirs():
    if LR_DIR_OVERRIDE and HR_DIR_OVERRIDE:
        lr_dir = Path(LR_DIR_OVERRIDE)
        hr_dir = Path(HR_DIR_OVERRIDE)
        data_root = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else common_root(lr_dir, hr_dir)
        return data_root, lr_dir, hr_dir, 'manual'

    for root in candidate_roots():
        all_dirs = [path for path in root.rglob('*') if path.is_dir()]
        all_dirs.append(root)
        by_parent_and_name = {(path.parent.resolve(), path.name.lower()): path for path in all_dirs}
        for lr_name, hr_name in NAME_PAIRS:
            for lr_dir in all_dirs:
                if lr_dir.name.lower() != lr_name or not has_images(lr_dir):
                    continue
                hr_dir = by_parent_and_name.get((lr_dir.parent.resolve(), hr_name))
                if hr_dir and has_images(hr_dir):
                    return common_root(lr_dir, hr_dir), lr_dir, hr_dir, f'{lr_name}->{hr_name}'
    raise FileNotFoundError(
        'Could not auto-detect LR/HR folders. Set LR_DIR_OVERRIDE and HR_DIR_OVERRIDE in the config cell.'
    )


def find_test_lr_dir():
    if TEST_LR_DIR_OVERRIDE:
        return Path(TEST_LR_DIR_OVERRIDE)
    for root in candidate_roots():
        all_dirs = [path for path in root.rglob('*') if path.is_dir()]
        preferred_names = {'lr', 'lq', '270p'}
        test_dirs = [path for path in all_dirs if 'test' in {part.lower() for part in path.parts}]
        for path in test_dirs:
            if path.name.lower() in preferred_names and has_images(path):
                return path
        for path in test_dirs:
            if has_images(path):
                return path
    return None


def find_sample_submission():
    if SAMPLE_SUBMISSION_OVERRIDE:
        return Path(SAMPLE_SUBMISSION_OVERRIDE)
    if not INPUT_ROOT.exists():
        return None
    matches = sorted(INPUT_ROOT.rglob('sample_submission*.csv'))
    return matches[0] if matches else None

DATA_ROOT, LR_DIR, HR_DIR, DETECTED_LAYOUT = find_pair_dirs()
TEST_LR_DIR = find_test_lr_dir()
SAMPLE_SUBMISSION = find_sample_submission()

print('Detected layout:', DETECTED_LAYOUT)
print('DATA_ROOT:', DATA_ROOT)
print('LR_DIR:', LR_DIR)
print('HR_DIR:', HR_DIR)
print('TEST_LR_DIR:', TEST_LR_DIR)
print('SAMPLE_SUBMISSION:', SAMPLE_SUBMISSION)

## 4. Build paired metadata

生成两份文件：

- `data/kaggle_pairs.csv` 给本项目的 baseline / metric 脚本使用
- `data/kaggle_meta_info_srgd_pair.txt` 给 Real-ESRGAN paired dataset 使用

In [ ]:
import pandas as pd

PAIRS_CSV = DATA_DIR / 'kaggle_pairs.csv'
META_INFO = DATA_DIR / 'kaggle_meta_info_srgd_pair.txt'

prepare_cmd = [
    sys.executable, PROJECT_DIR / 'scripts' / 'prepare_pairs.py',
    '--data-root', DATA_ROOT,
    '--lr-dir', LR_DIR,
    '--hr-dir', HR_DIR,
    '--out', PAIRS_CSV,
    '--meta-info', META_INFO,
]
if PAIR_LIMIT is not None:
    prepare_cmd += ['--limit', PAIR_LIMIT]

run(prepare_cmd, cwd=PROJECT_DIR)
pairs = pd.read_csv(PAIRS_CSV)
print('pairs:', len(pairs))
print(pairs['split'].value_counts(dropna=False))
display(pairs.head())

## 5. Bicubic baseline and metrics

先跑一个轻量 baseline。它能验证数据配对、输出目录和 PSNR/SSIM 评估是否正常。

In [ ]:
BICUBIC_DIR = OUTPUT_DIR / 'bicubic'
BICUBIC_METRICS = OUTPUT_DIR / 'bicubic_metrics.csv'
BICUBIC_SUMMARY = OUTPUT_DIR / 'bicubic_summary.json'

run([
    sys.executable, PROJECT_DIR / 'scripts' / 'run_bicubic.py',
    '--pairs', PAIRS_CSV,
    '--out-dir', BICUBIC_DIR,
    '--overwrite',
], cwd=PROJECT_DIR)

run([
    sys.executable, PROJECT_DIR / 'scripts' / 'evaluate_sr.py',
    '--pairs', PAIRS_CSV,
    '--sr-dir', BICUBIC_DIR,
    '--out', BICUBIC_METRICS,
    '--summary', BICUBIC_SUMMARY,
    '--resize-sr',
], cwd=PROJECT_DIR)

print(BICUBIC_SUMMARY.read_text())

## 6. Download Real-ESRGAN pretrained weights

微调需要 generator 和 discriminator 权重；推理只需要 generator 权重。

In [ ]:
from urllib.request import urlretrieve

PRETRAIN_DIR = REAL_ESRGAN_DIR / 'experiments' / 'pretrained_models'
PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS = {
    'RealESRGAN_x4plus.pth': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    'RealESRGAN_x4plus_netD.pth': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.3/RealESRGAN_x4plus_netD.pth',
}

for filename, url in WEIGHTS.items():
    target = PRETRAIN_DIR / filename
    if target.exists():
        print('exists:', target)
        continue
    print('downloading:', url)
    urlretrieve(url, target)
    print('saved:', target)

## 7. Pretrained Real-ESRGAN inference

这一步跑官方预训练模型，作为比 bicubic 更强的 baseline。`TILE = 256` 对 T4 通常比较稳，OOM 时改成 `128`。

In [ ]:
REALESRGAN_PRETRAINED_DIR = OUTPUT_DIR / 'realesrgan_x4plus_pretrained'
REALESRGAN_METRICS = OUTPUT_DIR / 'realesrgan_x4plus_pretrained_metrics.csv'
REALESRGAN_SUMMARY = OUTPUT_DIR / 'realesrgan_x4plus_pretrained_summary.json'

if RUN_PRETRAINED_INFERENCE:
    infer_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'run_realesrgan.py',
        '--realesrgan-dir', REAL_ESRGAN_DIR,
        '--pairs', PAIRS_CSV,
        '--out-dir', REALESRGAN_PRETRAINED_DIR,
        '--model-name', MODEL_NAME,
        '--outscale', 4,
        '--tile', TILE,
        '--gpu-id', 0,
    ]
    if INFERENCE_LIMIT is not None:
        infer_cmd += ['--limit', INFERENCE_LIMIT]
    run(infer_cmd, cwd=PROJECT_DIR)

    eval_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'evaluate_sr.py',
        '--pairs', PAIRS_CSV,
        '--sr-dir', REALESRGAN_PRETRAINED_DIR,
        '--out', REALESRGAN_METRICS,
        '--summary', REALESRGAN_SUMMARY,
        '--resize-sr',
    ]
    if INFERENCE_LIMIT is not None:
        eval_cmd += ['--limit', INFERENCE_LIMIT]
    run(eval_cmd, cwd=PROJECT_DIR)
    print(REALESRGAN_SUMMARY.read_text())
else:
    print('Skipped pretrained Real-ESRGAN inference.')

## 8. Visual comparison grid

输出 `LR | Bicubic | RealESRGAN | HR`，用于快速看模型是否真的在生成高分辨率图。

In [ ]:
from IPython.display import Image as IPyImage, display

GRID_DIR = OUTPUT_DIR / 'grids'
methods = [f'Bicubic={BICUBIC_DIR}']
if RUN_PRETRAINED_INFERENCE and REALESRGAN_PRETRAINED_DIR.exists():
    methods.append(f'RealESRGAN={REALESRGAN_PRETRAINED_DIR}')

run([
    sys.executable, PROJECT_DIR / 'scripts' / 'make_visual_grid.py',
    '--pairs', PAIRS_CSV,
    '--methods', *methods,
    '--out-dir', GRID_DIR,
    '--limit', GRID_LIMIT,
], cwd=PROJECT_DIR)

first_grid = sorted(GRID_DIR.glob('*_grid.png'))[0]
print(first_grid)
display(IPyImage(filename=str(first_grid)))

## 9. Generate the fine-tuning config

这里生成 Real-ESRGAN paired-data 微调配置。`num_gpu: auto` 会让 BasicSR/Real-ESRGAN 使用当前可见 GPU 数，下一格用 2 个进程启动 DDP。

In [ ]:
FINETUNE_CONFIG = CONFIG_DIR / 'kaggle_finetune_realesrgan_x4plus_pairdata.yml'

config_cmd = [
    sys.executable, PROJECT_DIR / 'scripts' / 'write_realesrgan_config.py',
    '--data-root', DATA_ROOT,
    '--meta-info', META_INFO,
    '--out', FINETUNE_CONFIG,
    '--batch-size', BATCH_SIZE_PER_GPU,
    '--workers', NUM_WORKERS_PER_GPU,
    '--gt-size', GT_SIZE,
    '--total-iter', TOTAL_ITER,
    '--save-freq', SAVE_FREQ,
    '--warmup-iter', WARMUP_ITER,
    '--d-init-iters', D_INIT_ITERS,
    '--enlarge-ratio', ENLARGE_RATIO,
    '--gan-weight', GAN_WEIGHT,
    '--prefetch-mode', '~',
    '--pretrain-g', 'experiments/pretrained_models/RealESRGAN_x4plus.pth',
    '--pretrain-d', 'experiments/pretrained_models/RealESRGAN_x4plus_netD.pth',
]
if USE_ROT:
    config_cmd += ['--use-rot']
else:
    config_cmd += ['--no-rot']

run(config_cmd, cwd=PROJECT_DIR)

print(FINETUNE_CONFIG)
print(FINETUNE_CONFIG.read_text()[:3000])

## 10. Fine-tune on T4 x2

这格会在 Kaggle 双 T4 上用 DDP 启动 Real-ESRGAN。默认 smoke 模式只跑 100 iter；全量实验请把第一格改成 `RUN_MODE = "full"` 并按显存调整 `BATCH_SIZE_PER_GPU`。

In [ ]:
if RUN_FINETUNE:
    import torch
    for _i in range(torch.cuda.device_count()):
        _free, _total = torch.cuda.mem_get_info(_i)
        print(f'GPU {_i}: {_free/1e9:.1f}/{_total/1e9:.1f} GB free  ({torch.cuda.get_device_name(_i)})')

    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = ','.join(str(i) for i in range(USE_GPUS))
    env.setdefault('OMP_NUM_THREADS', '2')
    # Kaggle T4 x2 VMs typically lack NVLink P2P; these flags prevent NCCL hangs.
    env.setdefault('NCCL_P2P_DISABLE', '1')
    env.setdefault('NCCL_IB_DISABLE', '1')

    if USE_GPUS >= 2:
        # torchrun replaces the deprecated torch.distributed.launch (removed in PyTorch 2.x).
        train_cmd = [
            'torchrun',
            f'--nproc_per_node={USE_GPUS}',
            '--master_port=4321',
            'realesrgan/train.py',
            '-opt', FINETUNE_CONFIG,
            '--launcher', 'pytorch',
            '--auto_resume',
        ]
    else:
        train_cmd = [
            sys.executable, 'realesrgan/train.py',
            '-opt', FINETUNE_CONFIG,
            '--auto_resume',
        ]

    try:
        run(train_cmd, cwd=REAL_ESRGAN_DIR, env=env)
    except subprocess.CalledProcessError as exc:
        print(f'\nTraining failed (exit {exc.returncode}).')
        print('Tip: if you see CUDA out-of-memory errors, set BATCH_SIZE_PER_GPU = 1 in Cell 1.')
        raise
else:
    print('Skipped fine-tuning.')

## 11. Locate the latest fine-tuned generator checkpoint

Real-ESRGAN 通常会把 checkpoint 放在 `Real-ESRGAN/experiments/<config name>/models`。

In [ ]:
MODEL_DIR = REAL_ESRGAN_DIR / 'experiments' / 'finetune_RealESRGANx4plus_SRGD_pairdata' / 'models'

def find_best_checkpoint(model_dir):
    if not model_dir.exists():
        return None
    # Prefer the EMA checkpoint — it's smoother and is what the official RealESRGAN
    # uses for inference (net_g_ema is the exponential moving average of the generator).
    ema = sorted(model_dir.glob('net_g_ema_*.pth'), key=lambda p: p.stat().st_mtime)
    if ema:
        return ema[-1]
    regular = sorted(model_dir.glob('net_g_[0-9]*.pth'), key=lambda p: p.stat().st_mtime)
    return regular[-1] if regular else None

all_checkpoints = sorted(MODEL_DIR.glob('net_g*.pth'), key=lambda p: p.stat().st_mtime) if MODEL_DIR.exists() else []
FINETUNED_G = find_best_checkpoint(MODEL_DIR)

print('MODEL_DIR:', MODEL_DIR)
print('All checkpoints:', [p.name for p in all_checkpoints])
print('Selected (EMA preferred):', FINETUNED_G)

## 12. Optional: inference with the fine-tuned checkpoint

如果微调已经保存了 `net_g_*.pth`，这里会用它重新跑一轮验证集/样本推理和指标。

In [ ]:
FINETUNED_DIR = OUTPUT_DIR / 'realesrgan_x4plus_finetuned'
FINETUNED_METRICS = OUTPUT_DIR / 'realesrgan_x4plus_finetuned_metrics.csv'
FINETUNED_SUMMARY = OUTPUT_DIR / 'realesrgan_x4plus_finetuned_summary.json'

if FINETUNED_G is not None:
    infer_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'run_realesrgan.py',
        '--realesrgan-dir', REAL_ESRGAN_DIR,
        '--pairs', PAIRS_CSV,
        '--out-dir', FINETUNED_DIR,
        '--model-name', MODEL_NAME,
        '--model-path', FINETUNED_G,
        '--outscale', 4,
        '--tile', TILE,
        '--gpu-id', 0,
    ]
    if INFERENCE_LIMIT is not None:
        infer_cmd += ['--limit', INFERENCE_LIMIT]
    run(infer_cmd, cwd=PROJECT_DIR)

    eval_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'evaluate_sr.py',
        '--pairs', PAIRS_CSV,
        '--sr-dir', FINETUNED_DIR,
        '--out', FINETUNED_METRICS,
        '--summary', FINETUNED_SUMMARY,
        '--resize-sr',
    ]
    if INFERENCE_LIMIT is not None:
        eval_cmd += ['--limit', INFERENCE_LIMIT]
    run(eval_cmd, cwd=PROJECT_DIR)
    print(FINETUNED_SUMMARY.read_text())
else:
    print('No fine-tuned generator checkpoint found yet.')

## 13. Course Metric Summary

For the course project, compute PSNR, SSIM, and LPIPS on the same subset and display a compact comparison table.


In [ ]:
import json
import pandas as pd

COURSE_METRIC_LIMIT = INFERENCE_LIMIT or COURSE_EVAL_LIMIT
COURSE_METHODS = {
    'Bicubic baseline': BICUBIC_DIR,
}
if REALESRGAN_PRETRAINED_DIR.exists():
    COURSE_METHODS['Real-ESRGAN pretrained'] = REALESRGAN_PRETRAINED_DIR
if FINETUNED_DIR.exists():
    COURSE_METHODS['Real-ESRGAN finetuned'] = FINETUNED_DIR

metric_rows = []
for method_name, sr_dir in COURSE_METHODS.items():
    safe_name = method_name.lower().replace(' ', '_').replace('-', '').replace('/', '_')
    summary_path = OUTPUT_DIR / f'{safe_name}_course_summary.json'
    metrics_path = OUTPUT_DIR / f'{safe_name}_course_metrics.csv'
    run([
        sys.executable, PROJECT_DIR / 'scripts' / 'evaluate_sr.py',
        '--pairs', PAIRS_CSV,
        '--sr-dir', sr_dir,
        '--out', metrics_path,
        '--summary', summary_path,
        '--resize-sr',
        '--limit', COURSE_METRIC_LIMIT,
        '--lpips',
        '--dists',
    ], cwd=PROJECT_DIR)
    data = json.loads(summary_path.read_text())
    metric_rows.append({
        'Method': method_name,
        'Images': data['count'],
        'PSNR (higher is better)': round(data['psnr_mean'], 4),
        'SSIM (higher is better)': round(data['ssim_mean'], 4),
        'LPIPS (lower is better)': round(data['lpips_mean'], 4),
        'DISTS (lower is better)': round(data['dists_mean'], 4) if data.get('dists_mean') is not None else None,
    })

metric_table = pd.DataFrame(metric_rows)
display(metric_table)
metric_table.to_csv(OUTPUT_DIR / 'course_metric_summary.csv', index=False)
print('Saved:', OUTPUT_DIR / 'course_metric_summary.csv')

## 14. Optional Competition Submission, Not Needed For Course

如果数据集中有 `test/lr` 或类似目录，这里会对 test LR 图片推理，并生成比赛要求的 `submission.csv`。

In [ ]:
TEST_OUTPUT_DIR = OUTPUT_DIR / 'test_realesrgan'
SUBMISSION_CSV = WORK_ROOT / 'submission.csv'

if RUN_TEST_SUBMISSION and TEST_LR_DIR is not None:
    checkpoint_for_submission = FINETUNED_G if FINETUNED_G is not None else None
    model_args = ['--model_path', str(checkpoint_for_submission)] if checkpoint_for_submission else []

    # Split test images over the available GPUs for faster inference.
    staged_root = OUTPUT_DIR / 'test_lr_chunks'
    if staged_root.exists():
        shutil.rmtree(staged_root)
    TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    test_images = sorted(path for path in TEST_LR_DIR.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    print('test images:', len(test_images))

    processes = []
    for gpu_index in range(USE_GPUS):
        chunk_dir = staged_root / f'gpu{gpu_index}'
        chunk_dir.mkdir(parents=True, exist_ok=True)
        for image_path in test_images[gpu_index::USE_GPUS]:
            target = chunk_dir / image_path.name
            if not target.exists():
                shutil.copy2(image_path, target)

        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = str(gpu_index)
        cmd = [
            sys.executable, 'inference_realesrgan.py',
            '-n', MODEL_NAME,
            '-i', chunk_dir,
            '-o', TEST_OUTPUT_DIR,
            '--outscale', '4',
            '--suffix', '',
            '--tile', str(TILE),
            '--ext', 'png',
            *model_args,
        ]
        print('$ ' + ' '.join(str(part) for part in cmd))
        processes.append(subprocess.Popen([str(part) for part in cmd], cwd=REAL_ESRGAN_DIR, env=env))

    for process in processes:
        if process.wait() != 0:
            raise RuntimeError('One of the Real-ESRGAN test inference workers failed.')

    submit_cmd = [
        sys.executable, PROJECT_DIR / 'scripts' / 'make_kaggle_submission.py',
        '--images-dir', TEST_OUTPUT_DIR,
        '--out', SUBMISSION_CSV,
    ]
    if SAMPLE_SUBMISSION is not None:
        submit_cmd += ['--sample-submission', SAMPLE_SUBMISSION]
    run(submit_cmd, cwd=PROJECT_DIR)
    print('Submission:', SUBMISSION_CSV)
else:
    print('Skipped submission generation. TEST_LR_DIR was not found or RUN_TEST_SUBMISSION=False.')